# Module 04 — Exercises: The Training Loop (FashionMNIST)
FashionMNIST has the same format as MNIST (28×28 grayscale images, 10 classes) but shows clothing, and it is
harder. Write every piece of the pipeline yourself.

In [ ]:
# Standard setup: make the repo root importable, pick a device, fix the random seed.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]  # notebooks live in modules/XX_name/, so go up two levels
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from common.device import get_device, seed_everything, fast_dev_run

FAST_DEV_RUN = fast_dev_run()  # True only for automated checks; leave it False when learning
DATA_DIR = ROOT / "data"
device = get_device()
seed_everything(42)

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torchvision import datasets, transforms

CLASSES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Boot"]

### Exercise 1 — A custom Dataset
Write `NoisyRegression`, a `Dataset` that generates `n` points with `x ~ U(-3, 3)` and `y = sin(x) + noise`
(noise std 0.1), **once** in `__init__`. Each item should be `(x, y)` where both are float tensors of shape (1,).

In [ ]:
class NoisyRegression(Dataset):
    def __init__(self, n, seed=0):
        g = torch.Generator().manual_seed(seed)
        # TODO: create self.x and self.y
        raise NotImplementedError

    def __len__(self):
        # TODO
        raise NotImplementedError

    def __getitem__(self, i):
        # TODO
        raise NotImplementedError

In [ ]:
ds = NoisyRegression(300)
xb, yb = next(iter(DataLoader(ds, batch_size=32)))
assert len(ds) == 300 and xb.shape == (32, 1) and yb.shape == (32, 1) and xb.dtype == torch.float32
assert xb.min() >= -3 and xb.max() <= 3
assert torch.equal(NoisyRegression(300)[5][0], ds[5][0]), "data should be generated once, deterministically"
print("Exercise 1 passed ✅")

### Exercise 2 — Fit it with an MLP (a regression loop)
Build an `nn.Sequential` with one hidden layer of 64 units and `Tanh`, and train it with `nn.MSELoss` and
`torch.optim.Adam(lr=1e-2)` for 200 epochs over a `DataLoader(ds, batch_size=32, shuffle=True)`.
Reach a final MSE on the whole dataset below 0.02.

In [ ]:
torch.manual_seed(0)
reg_model = None  # TODO
# TODO: training loop

In [ ]:
with torch.no_grad():
    final = nn.functional.mse_loss(reg_model(ds.x), ds.y).item()
print("final MSE:", round(final, 4))
assert final < 0.02
print("Exercise 2 passed ✅")

### Exercise 3 — FashionMNIST loaders
Create `train_loader`, `val_loader`, `test_loader`:
- transform: `ToTensor()` then `Normalize((0.2860,), (0.3530,))`;
- split the 60k training images into 54k train / 6k validation with a seeded generator;
- batch size 128 with shuffling for train; 512 without shuffling for val/test.
(`FAST_DEV_RUN` shrinks the datasets afterwards; that code is already written.)

In [ ]:
# TODO: train_ds, val_ds, test_ds
if FAST_DEV_RUN:
    train_ds, val_ds, test_ds = Subset(train_ds, range(3000)), Subset(val_ds, range(500)), Subset(test_ds, range(500))
# TODO: the three DataLoaders

In [ ]:
xb, yb = next(iter(train_loader))
assert xb.shape == (128, 1, 28, 28) and yb.dtype == torch.long
assert abs(xb.mean().item()) < 0.2, "did you normalize?"
if not FAST_DEV_RUN:
    assert len(train_loader.dataset) == 54_000 and len(val_loader.dataset) == 6_000
print("Exercise 3 passed ✅")

### Exercise 4 — The model
Write `FashionMLP(nn.Module)`: flatten → Linear(784, 512) → ReLU → Linear(512, 256) → ReLU → Linear(256, 10).

In [ ]:
class FashionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
m = FashionMLP()
assert m(torch.randn(3, 1, 28, 28)).shape == (3, 10)
assert sum(p.numel() for p in m.parameters()) == 784 * 512 + 512 + 512 * 256 + 256 + 256 * 10 + 10
print("Exercise 4 passed ✅")

### Exercise 5 — Train and evaluate loops
Write `train_epoch(model, loader, opt)` (returns the mean training loss) and `accuracy(model, loader)` (returns
the fraction correct). Use `nn.functional.cross_entropy`. Remember `train()`/`eval()`, `no_grad`, and the device.

In [ ]:
def train_epoch(model, loader, opt):
    # TODO
    raise NotImplementedError


@torch.no_grad()
def accuracy(model, loader):
    # TODO
    raise NotImplementedError

Now train `FashionMLP` with Adam (lr 1e-3) for 8 epochs, printing the validation accuracy each epoch.
Target: **≥ 87% test accuracy** (88–89% is typical for an MLP after 8 epochs).

In [ ]:
torch.manual_seed(0)
model = None  # TODO: create, move to device, train

In [ ]:
test_acc = accuracy(model, test_loader)
print("test accuracy:", round(test_acc, 4))
assert test_acc >= (0.6 if FAST_DEV_RUN else 0.87)
print("Exercise 5 passed ✅")

### Exercise 6 — Save, load and predict
Save `model`'s state_dict to `checkpoints/fashion_mlp.pt`, load it into a **new** `FashionMLP`, and write
`predict(model, imgs)` that returns a list of class **names** for a batch of images.

In [ ]:
def predict(model, imgs):
    # TODO
    raise NotImplementedError


path = ROOT / "checkpoints" / "fashion_mlp.pt"
loaded = None  # TODO

In [ ]:
imgs, labels = next(iter(test_loader))
names = predict(loaded, imgs[:8])
assert names == predict(model, imgs[:8]) and all(n in CLASSES for n in names)
assert loaded is not model
print(list(zip(names, [CLASSES[i] for i in labels[:8]])))
print("Exercise 6 passed ✅")